In [ ]:
import numpy as np
import cartopy.feature as cfeature
from shapely.geometry import Point, LineString, box

In [16]:
# ── Islands / obstacles: (lon, lat, label, radius_deg_approx) ───────────────
OBSTACLES = [
    (-87.357, 45.184, "Chambers Island",   0.055),
    (-87.265, 45.159, "Adventure Island",  0.010),
    (-87.210, 45.179, "Horseshoe\nIsland", 0.005),
    (-87.497, 45.059, "Green Island",      0.013),
]

# ── Map extent: [lon_min, lon_max, lat_min, lat_max] ────────────────────────
EXTENT = [-87.80, -87.10, 44.88, 45.35]

land = cfeature.LAND.with_scale('10m')
bbox = box(EXTENT[0], EXTENT[2], EXTENT[1], EXTENT[3])

coast_polys = []
for geom in land.geometries():
    clipped = geom.intersection(bbox)
    if not clipped.is_empty:
        coast_polys.append(clipped)

# Extract exterior coordinates from each polygon
COAST_VERTICES = []
for poly in coast_polys:
    if hasattr(poly, 'exterior'):
        lons, lats = poly.exterior.xy
        COAST_VERTICES.extend([(lon, lat) for lon, lat in zip(lons, lats)])

In [ ]:
coastline = LineString(COAST_VERTICES)   # Extracted points

def coast_penalty(lat, lon, min_dist_nm=0.5, n=2):
    pt = Point(lon, lat)
    d_deg = coastline.distance(pt)
    d_nm  = d_deg * 60 * np.cos(np.deg2rad(lat))
    return 1.0 / np.maximum(d_nm, min_dist_nm)**n

In [ ]:
def island_penalty_log(lat, lon, center_lat, center_lon, a_nm, b_nm):
    """
    Smooth barrier function — goes to +inf as (lat,lon) approaches island center.
    a_nm, b_nm = semi-axes in nautical miles (lon-axis, lat-axis).
    """
    dlat = (lat - center_lat) * 60          # degrees to nm
    dlon = (lon - center_lon) * 60 * np.cos(np.deg2rad(center_lat))
    r = np.sqrt((dlon/a_nm)**2 + (dlat/b_nm)**2)  # normalised distance

    return -np.log(np.maximum(r - 1, 1e-6)) * (r < 1.5)

def island_penalty_inverse(lat, lon, center_lat, center_lon, a_nm, b_nm, n):
    dlat = (lat - center_lat) * 60          # degrees to nm
    dlon = (lon - center_lon) * 60 * np.cos(np.deg2rad(center_lat))
    r = np.sqrt((dlon/a_nm)**2 + (dlat/b_nm)**2)  # normalised distance

    return 1 / (r**n)